# 第8回: SARNN - Spatial Attention RNN
**深層予測学習（Deep Predictive Learning）入門**

## 1. はじめに

### CNNRNNの限界とSARNNの動機

第7回のCNNRNNは画像と関節角度を統合して学習できましたが、
**モデルが画像のどこに注目しているか** が不透明でした。

**SARNN (Spatial Attention RNN)** は、画像から **注意点座標** を明示的に抽出し、
タスクに重要な位置（物体やアームの位置など）と関節角度の時系列関係を学習します。

### SARNNのアーキテクチャ

```
入力画像 ──→ [Positional Encoder] ──→ 注意点座標 (x,y) ──┐
    │                                                    ├→ [LSTM] → 関節角度デコーダ → 予測関節角度
    │         入力関節角度 ────────────────────────────────┘     │
    │                                                          ├→ 注意点デコーダ → [InverseSpatialSoftmax]
    └──→ [Image Encoder] ──→ 特徴マップ ──→ × ヒートマップ ──→ [Image Decoder] → 予測画像
```

**ポイント:**
- Positional Encoder: SpatialSoftmax で画像から (x,y) 座標を抽出
- Image Encoder: 画像の特徴マップを抽出
- LSTM: 注意点 + 関節角度から次の状態を予測
- InverseSpatialSoftmax: 予測した注意点からヒートマップを生成
- ヒートマップ × 特徴マップ → 画像デコーダ

## 2. 環境セットアップ

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import os, json

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
torch.manual_seed(42)
np.random.seed(42)

## 3. SpatialSoftmax と InverseSpatialSoftmax

### SpatialSoftmax
CNN特徴マップの各チャネルに softmax を適用し、期待値として (x, y) 座標を抽出します。

$$\text{key}_x = \sum_{i,j} \text{pos}_x(i,j) \cdot \text{softmax}(\text{feat}(i,j) / \tau)$$

### InverseSpatialSoftmax
(x, y) 座標からガウシアンヒートマップを生成します。

$$\text{heatmap}(i,j) = \exp\left(-\frac{(\text{pos}(i,j) - \text{key})^2}{\sigma}\right)$$

In [ ]:
class SpatialSoftmax(nn.Module):
    """CNNの特徴マップからXY座標を抽出"""
    def __init__(self, width, height, temperature=1e-4):
        super().__init__()
        self.temperature = temperature
        pos_x, pos_y = np.meshgrid(
            np.linspace(0, 1, height), np.linspace(0, 1, width), indexing="xy")
        self.register_buffer("pos_x", torch.from_numpy(pos_x.reshape(-1)).float())
        self.register_buffer("pos_y", torch.from_numpy(pos_y.reshape(-1)).float())

    def forward(self, x):
        B, C, W, H = x.shape
        logit = x.reshape(B, C, -1)
        att = torch.softmax(logit / self.temperature, dim=-1)
        ex = torch.sum(self.pos_x * att, dim=-1, keepdim=True)
        ey = torch.sum(self.pos_y * att, dim=-1, keepdim=True)
        keys = torch.cat([ex, ey], -1).reshape(B, C, 2)
        return keys, att.reshape(B, C, W, H)


class InverseSpatialSoftmax(nn.Module):
    """XY座標からガウシアンヒートマップを生成"""
    def __init__(self, width, height, heatmap_size=0.1):
        super().__init__()
        self.heatmap_size = heatmap_size
        pos_x, pos_y = np.meshgrid(
            np.linspace(0, 1, height), np.linspace(0, 1, width), indexing="xy")
        self.register_buffer("pos_xy",
            torch.from_numpy(np.stack([pos_x, pos_y], axis=0)).float())

    def forward(self, keys):
        d = torch.sum(
            torch.pow(self.pos_xy[None, None] - keys[:, :, :, None, None], 2.0), axis=2)
        return torch.exp(-d / self.heatmap_size)

print("SpatialSoftmax / InverseSpatialSoftmax 定義完了")

## 4. SARNN モデル

SARNNは以下の5つの出力を返します：
1. `y_image`: 予測画像
2. `y_joint`: 予測関節角度
3. `enc_pts`: エンコードされた注意点座標
4. `dec_pts`: デコードされた注意点座標
5. `rnn_hid`: LSTM隠れ状態

In [ ]:
class SARNN(nn.Module):
    def __init__(self, rec_dim, k_dim=5, joint_dim=7, temperature=1e-4,
                 heatmap_size=0.1, kernel_size=3, im_size=[64, 64]):
        super().__init__()
        self.k_dim = k_dim
        activation = nn.LeakyReLU(negative_slope=0.3)

        sub_im_size = [im_size[0] - 3*(kernel_size-1),
                       im_size[1] - 3*(kernel_size-1)]

        # Positional Encoder: 画像 → 注意点座標
        self.pos_encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, 1, 0), activation,
            nn.Conv2d(16, 32, 3, 1, 0), activation,
            nn.Conv2d(32, k_dim, 3, 1, 0), activation,
            SpatialSoftmax(width=sub_im_size[0], height=sub_im_size[1],
                          temperature=temperature),
        )

        # Image Encoder: 画像 → 特徴マップ
        self.im_encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, 1, 0), activation,
            nn.Conv2d(16, 32, 3, 1, 0), activation,
            nn.Conv2d(32, k_dim, 3, 1, 0), activation,
        )

        # LSTM: 注意点 + 関節角度 → 隠れ状態
        rec_in = joint_dim + k_dim * 2
        self.rec = nn.LSTMCell(rec_in, rec_dim)

        # Joint Decoder
        self.decoder_joint = nn.Sequential(
            nn.Linear(rec_dim, joint_dim), activation)

        # Point Decoder
        self.decoder_point = nn.Sequential(
            nn.Linear(rec_dim, k_dim * 2), activation)

        # Inverse Spatial Softmax: 注意点 → ヒートマップ
        self.issm = InverseSpatialSoftmax(
            width=sub_im_size[0], height=sub_im_size[1],
            heatmap_size=heatmap_size)

        # Image Decoder: ヒートマップ × 特徴マップ → 画像
        self.decoder_image = nn.Sequential(
            nn.ConvTranspose2d(k_dim, 32, 3, 1, 0), activation,
            nn.ConvTranspose2d(32, 16, 3, 1, 0), activation,
            nn.ConvTranspose2d(16, 3, 3, 1, 0), activation,
        )

        self.apply(self._weights_init)

    def _weights_init(self, m):
        if isinstance(m, nn.LSTMCell):
            nn.init.xavier_uniform_(m.weight_ih)
            nn.init.orthogonal_(m.weight_hh)
            nn.init.zeros_(m.bias_ih)
            nn.init.zeros_(m.bias_hh)
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, xi, xv, state=None):
        # 1. 画像エンコード
        im_hid = self.im_encoder(xi)           # 特徴マップ
        enc_pts, _ = self.pos_encoder(xi)       # 注意点座標
        enc_pts = enc_pts.reshape(-1, self.k_dim * 2)

        # 2. LSTM
        hid = torch.cat([enc_pts, xv], -1)     # 注意点 + 関節角度
        rnn_hid = self.rec(hid, state)

        # 3. デコード
        y_joint = self.decoder_joint(rnn_hid[0])
        dec_pts = self.decoder_point(rnn_hid[0])

        # 4. ヒートマップ生成 → 画像デコード
        dec_pts_in = dec_pts.reshape(-1, self.k_dim, 2)
        heatmap = self.issm(dec_pts_in)
        hid = torch.mul(heatmap, im_hid)       # ヒートマップ × 特徴マップ
        y_image = self.decoder_image(hid)

        return y_image, y_joint, enc_pts, dec_pts, rnn_hid

# 動作確認
model = SARNN(rec_dim=50, k_dim=5, joint_dim=7).to(device)
dummy_img = torch.randn(2, 3, 64, 64, device=device)
dummy_joint = torch.randn(2, 7, device=device)
y_img, y_jnt, enc_p, dec_p, _ = model(dummy_img, dummy_joint)
print(f"予測画像: {y_img.shape}")
print(f"予測関節: {y_jnt.shape}")
print(f"エンコード注意点: {enc_p.shape}")
print(f"デコード注意点: {dec_p.shape}")
print(f"パラメータ数: {sum(p.numel() for p in model.parameters()):,}")

## 5. データセット (LeRobot形式)

第4回で作成したLeRobot形式のデータセットを使用します。
ここではデモ用にダミーデータを生成します。

In [ ]:
def create_dummy_data(num_episodes=20, T=50, img_size=64, joint_dim=7):
    images_all, joints_all = [], []
    for ep in range(num_episodes):
        t = np.linspace(0, 4*np.pi, T)
        joints = np.stack([0.5*np.sin(t + i*0.5) for i in range(joint_dim)], axis=-1)
        joints += np.random.normal(0, 0.02, joints.shape)
        joints_all.append(joints.astype(np.float32))

        imgs = []
        for step in range(T):
            img = np.ones((img_size, img_size, 3), dtype=np.float32) * 0.9
            cx = int((joints[step, 0]*0.4 + 0.5) * img_size)
            cy = int((joints[step, 1]*0.4 + 0.5) * img_size)
            cx, cy = np.clip(cx, 3, img_size-4), np.clip(cy, 3, img_size-4)
            img[cy-3:cy+3, cx-3:cx+3] = [0.1, 0.1, 0.9]
            imgs.append(img)
        images_all.append(np.array(imgs))

    return np.array(images_all), np.array(joints_all)

images_np, joints_np = create_dummy_data()
# (N, T, C, H, W) 形式に変換
images_t = torch.from_numpy(images_np.transpose(0, 1, 4, 2, 3)).to(device)
joints_t = torch.from_numpy(joints_np).to(device)
print(f"画像: {images_t.shape}, 関節: {joints_t.shape}")

## 6. BPTT (Backpropagation Through Time)

### Full BPTT for SARNN

SARNNの学習では **3種類の損失** を組み合わせます：

1. **画像損失** (MSE): 予測画像と正解画像
2. **関節損失** (MSE): 予測関節角度と正解関節角度
3. **注意点損失** (MSE + annealing): エンコーダ注意点とデコーダ注意点の一致

注意点損失には **アニーリング**（徐々に重みを増加）を適用します。
学習初期は画像・関節の学習に集中し、後半で注意点の整合性を高めます。

In [ ]:
class fullBPTTtrainer_sarnn:
    def __init__(self, model, optimizer, loss_weights=[1.0, 1.0, 0.1], device="cpu"):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.loss_weights = loss_weights
        self.device = device
        self.epoch_count = 0

    def process_epoch(self, images, joints, training=True):
        if training:
            self.model.train()
        else:
            self.model.eval()

        T = images.shape[1]
        state = None
        yi_list, yv_list = [], []
        enc_list, dec_list = [], []

        self.optimizer.zero_grad()

        for t in range(T - 1):
            y_img, y_jnt, enc_p, dec_p, state = self.model(
                images[:, t], joints[:, t], state)
            yi_list.append(y_img)
            yv_list.append(y_jnt)
            enc_list.append(enc_p)
            dec_list.append(dec_p)

        yi_hat = torch.stack(yi_list, dim=1)
        yv_hat = torch.stack(yv_list, dim=1)

        # 1. 画像損失
        img_loss = nn.MSELoss()(yi_hat, images[:, 1:]) * self.loss_weights[0]

        # 2. 関節損失
        joint_loss = nn.MSELoss()(yv_hat, joints[:, 1:]) * self.loss_weights[1]

        # 3. 注意点損失 (annealing: 徐々に重みを増加)
        pt_weight = min(self.epoch_count / 1000.0, 1.0) * self.loss_weights[2]
        pt_loss = nn.MSELoss()(
            torch.stack(dec_list[:-1]), torch.stack(enc_list[1:])) * pt_weight

        loss = img_loss + joint_loss + pt_loss

        if training:
            loss.backward()
            self.optimizer.step()
            self.epoch_count += 1

        return loss.item(), img_loss.item(), joint_loss.item(), pt_loss.item()

print("fullBPTTtrainer_sarnn 定義完了")

## 7. Truncated BPTT

### Full BPTT vs Truncated BPTT

**Full BPTT** は系列全体を通して勾配を計算しますが、
長い系列ではメモリ消費が大きく、勾配消失/爆発が問題になります。

**Truncated BPTT** は系列をウィンドウに分割し、各ウィンドウ内でのみ逆伝播を行います。
ウィンドウ間では隠れ状態を **detach** して勾配の伝播を打ち切ります。

```
Full BPTT:    [t=0 ─────────────────────── t=T]  ← 全体で逆伝播

Truncated:    [t=0 ── t=W] [t=W ── t=2W] [t=2W ── t=T]
                  ↑           ↑              ↑
               逆伝播      逆伝播         逆伝播
              (detach)    (detach)
```

In [ ]:
class truncatedBPTTtrainer:
    def __init__(self, model, optimizer, window_size=25, window_step=25,
                 loss_weights=[1.0, 1.0, 0.1], device="cpu"):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.window_size = window_size
        self.window_step = window_step
        self.loss_weights = loss_weights
        self.device = device
        self.epoch_count = 0

    def process_epoch(self, images, joints, training=True):
        if training:
            self.model.train()
        else:
            self.model.eval()

        T = images.shape[1] - 1
        state = None
        total_loss = 0.0
        n_windows = 0

        for start in range(0, T, self.window_step):
            end = min(start + self.window_step, T)

            yi_list, yv_list = [], []
            enc_list, dec_list = [], []

            for t in range(start, end):
                y_img, y_jnt, enc_p, dec_p, state = self.model(
                    images[:, t], joints[:, t], state)
                yi_list.append(y_img)
                yv_list.append(y_jnt)
                enc_list.append(enc_p)
                dec_list.append(dec_p)

            # ★ ウィンドウ間で隠れ状態をdetach
            if end != T:
                state = tuple(s.detach() for s in state)

            yi_hat = torch.stack(yi_list, dim=1)
            yv_hat = torch.stack(yv_list, dim=1)

            img_loss = nn.MSELoss()(yi_hat, images[:, start+1:end+1]) * self.loss_weights[0]
            joint_loss = nn.MSELoss()(yv_hat, joints[:, start+1:end+1]) * self.loss_weights[1]

            pt_weight = min(self.epoch_count / 1000.0, 1.0) * self.loss_weights[2]
            if len(dec_list) > 1:
                pt_loss = nn.MSELoss()(
                    torch.stack(dec_list[:-1]), torch.stack(enc_list[1:])) * pt_weight
            else:
                pt_loss = torch.tensor(0.0)

            loss = img_loss + joint_loss + pt_loss
            total_loss += loss.item()
            n_windows += 1

            if training:
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

        self.epoch_count += 1
        return total_loss / max(n_windows, 1)

print("truncatedBPTTtrainer 定義完了")

## 8. Robomimicデータを使った学習

In [ ]:
# Full BPTT で学習
model = SARNN(rec_dim=50, k_dim=5, joint_dim=7).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
trainer = fullBPTTtrainer_sarnn(model, optimizer, loss_weights=[1.0, 1.0, 0.1], device=device)

num_epochs = 100
history = {"total": [], "image": [], "joint": [], "point": []}

for epoch in range(num_epochs):
    total, img_l, jnt_l, pt_l = trainer.process_epoch(images_t, joints_t)
    history["total"].append(total)
    history["image"].append(img_l)
    history["joint"].append(jnt_l)
    history["point"].append(pt_l)
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}: total={total:.4f}, img={img_l:.4f}, "
              f"joint={jnt_l:.4f}, point={pt_l:.4f}")
print("学習完了！")

In [ ]:
# 損失曲線
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, key, title in zip(axes, ["image", "joint", "point"],
                           ["画像損失", "関節損失", "注意点損失"]):
    ax.plot(history[key])
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title(title)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. 予測結果の可視化

### 入力画像、予測画像、予測注意点、関節角度

In [ ]:
# 推論
model.eval()
idx = 0
test_img = images_t[idx:idx+1]
test_jnt = joints_t[idx:idx+1]

pred_imgs, pred_joints = [], []
enc_pts_all, dec_pts_all = [], []
state = None

with torch.no_grad():
    for t in range(test_img.shape[1] - 1):
        y_img, y_jnt, enc_p, dec_p, state = model(test_img[:, t], test_jnt[:, t], state)
        pred_imgs.append(y_img[0].cpu().numpy().transpose(1, 2, 0))
        pred_joints.append(y_jnt[0].cpu().numpy())
        enc_pts_all.append(enc_p[0].cpu().numpy())
        dec_pts_all.append(dec_p[0].cpu().numpy())

pred_joints = np.array(pred_joints)
gt_joints = test_jnt[0, 1:].cpu().numpy()

In [ ]:
# 入力画像 vs 予測画像 + 注意点
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
frames = np.linspace(0, len(pred_imgs)-1, 5, dtype=int)

for i, f in enumerate(frames):
    # 入力画像
    gt = test_img[0, f+1].cpu().numpy().transpose(1, 2, 0)
    axes[0, i].imshow(np.clip(gt, 0, 1))
    axes[0, i].set_title(f"入力 t={f+1}")
    axes[0, i].axis("off")

    # 予測画像 + エンコーダ注意点 (緑)
    pred = np.clip(pred_imgs[f], 0, 1)
    axes[1, i].imshow(pred)
    enc_p = enc_pts_all[f].reshape(-1, 2)
    axes[1, i].scatter(enc_p[:, 0]*64, enc_p[:, 1]*64, c="lime", s=40, marker="o", label="enc")
    axes[1, i].set_title(f"予測 t={f+1}")
    axes[1, i].axis("off")

    # 予測画像 + デコーダ注意点 (赤)
    axes[2, i].imshow(pred)
    dec_p = dec_pts_all[f].reshape(-1, 2)
    axes[2, i].scatter(dec_p[:, 0]*64, dec_p[:, 1]*64, c="red", s=40, marker="x", label="dec")
    axes[2, i].set_title(f"dec注意点 t={f+1}")
    axes[2, i].axis("off")

axes[0, 0].set_ylabel("正解", fontsize=12)
axes[1, 0].set_ylabel("予測+enc点", fontsize=12)
axes[2, 0].set_ylabel("予測+dec点", fontsize=12)
plt.suptitle("SARNN: 入力画像 vs 予測画像 vs 注意点", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 関節角度の予測
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()
for i in range(7):
    ax = axes[i]
    ax.plot(gt_joints[:, i], "b-", label="正解", linewidth=1.5)
    ax.plot(pred_joints[:, i], "r--", label="予測", linewidth=1.5)
    ax.set_title(f"Joint {i}")
    ax.set_xlabel("Time step")
    ax.grid(True, alpha=0.3)
    if i == 0: ax.legend()
for i in range(7, 9): axes[i].set_visible(False)
plt.suptitle("関節角度: 正解 vs 予測", fontsize=14)
plt.tight_layout()
plt.show()

### 演習1: SARNNのforward関数を実装せよ

SpatialSoftmax → LSTM → InverseSpatialSoftmax の流れを理解し、
以下のSARNNクラスのforward関数を完成させてください。

In [ ]:
class MySARNN(nn.Module):
    def __init__(self, rec_dim=50, k_dim=5, joint_dim=7, im_size=[64,64]):
        super().__init__()
        self.k_dim = k_dim
        act = nn.LeakyReLU(0.3)
        sub = [im_size[0]-6, im_size[1]-6]
        self.pos_encoder = nn.Sequential(
            nn.Conv2d(3,16,3,1,0), act, nn.Conv2d(16,32,3,1,0), act,
            nn.Conv2d(32,k_dim,3,1,0), act,
            SpatialSoftmax(sub[0], sub[1]))
        self.im_encoder = nn.Sequential(
            nn.Conv2d(3,16,3,1,0), act, nn.Conv2d(16,32,3,1,0), act,
            nn.Conv2d(32,k_dim,3,1,0), act)
        self.rec = nn.LSTMCell(joint_dim + k_dim*2, rec_dim)
        self.dec_joint = nn.Sequential(nn.Linear(rec_dim, joint_dim), act)
        self.dec_point = nn.Sequential(nn.Linear(rec_dim, k_dim*2), act)
        self.issm = InverseSpatialSoftmax(sub[0], sub[1])
        self.dec_image = nn.Sequential(
            nn.ConvTranspose2d(k_dim,32,3,1,0), act,
            nn.ConvTranspose2d(32,16,3,1,0), act,
            nn.ConvTranspose2d(16,3,3,1,0), act)

    def forward(self, xi, xv, state=None):
        # TODO: ここを実装
        # ヒント: im_encoder, pos_encoder, cat, rec, dec_joint, dec_point, issm, mul, dec_image
        pass

# m = MySARNN().to(device)
# y_img, y_jnt, enc_p, dec_p, state = m(dummy_img, dummy_joint)

<details>
<summary><b>回答例を表示</b></summary>

```python
def forward(self, xi, xv, state=None):
    im_hid = self.im_encoder(xi)
    enc_pts, _ = self.pos_encoder(xi)
    enc_pts = enc_pts.reshape(-1, self.k_dim * 2)
    hid = torch.cat([enc_pts, xv], -1)

    rnn_hid = self.rec(hid, state)
    y_joint = self.dec_joint(rnn_hid[0])
    dec_pts = self.dec_point(rnn_hid[0])

    dec_pts_in = dec_pts.reshape(-1, self.k_dim, 2)
    heatmap = self.issm(dec_pts_in)
    hid = torch.mul(heatmap, im_hid)
    y_image = self.dec_image(hid)

    return y_image, y_joint, enc_pts, dec_pts, rnn_hid
```

</details>

### 演習2: fullBPTTtrainerのpoint_lossを含むprocess_epochを実装せよ

3つの損失（画像、関節、注意点）を持つprocess_epochを実装してください。
注意点損失にはアニーリング（epoch_countに基づく重み増加）を適用してください。

In [ ]:
class MyTrainer:
    def __init__(self, model, optimizer, loss_weights=[1.0, 1.0, 0.1], device="cpu"):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.loss_weights = loss_weights
        self.device = device
        self.epoch_count = 0

    def process_epoch(self, images, joints):
        # TODO: 実装
        # 1. 時間ループで予測を蓄積
        # 2. 3つの損失を計算
        # 3. 注意点損失にアニーリングを適用
        pass

<details>
<summary><b>回答例を表示</b></summary>

```python
def process_epoch(self, images, joints):
    self.model.train()
    T = images.shape[1]
    state = None
    yi_list, yv_list, enc_list, dec_list = [], [], [], []
    self.optimizer.zero_grad()

    for t in range(T - 1):
        y_img, y_jnt, enc_p, dec_p, state = self.model(
            images[:, t], joints[:, t], state)
        yi_list.append(y_img)
        yv_list.append(y_jnt)
        enc_list.append(enc_p)
        dec_list.append(dec_p)

    yi_hat = torch.stack(yi_list, dim=1)
    yv_hat = torch.stack(yv_list, dim=1)

    img_loss = nn.MSELoss()(yi_hat, images[:, 1:]) * self.loss_weights[0]
    joint_loss = nn.MSELoss()(yv_hat, joints[:, 1:]) * self.loss_weights[1]
    pt_weight = min(self.epoch_count / 1000.0, 1.0) * self.loss_weights[2]
    pt_loss = nn.MSELoss()(
        torch.stack(dec_list[:-1]), torch.stack(enc_list[1:])) * pt_weight

    loss = img_loss + joint_loss + pt_loss
    loss.backward()
    self.optimizer.step()
    self.epoch_count += 1
    return loss.item()
```

</details>

### 演習3: TruncatedBPTTのwindow分割とstate detachを実装せよ

系列をwindow_step単位で分割し、ウィンドウ間では隠れ状態をdetachする
TruncatedBPTTのprocess_epochを実装してください。

ヒント: `state = tuple(s.detach() for s in state)` でdetachできます。

In [ ]:
class MyTruncatedTrainer:
    def __init__(self, model, optimizer, window_step=25, device="cpu"):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.window_step = window_step
        self.device = device

    def process_epoch(self, images, joints):
        # TODO: 実装
        # 1. range(0, T, window_step) でウィンドウを分割
        # 2. 各ウィンドウ内で予測・損失計算・逆伝播
        # 3. ウィンドウ間で state を detach
        pass

<details>
<summary><b>回答例を表示</b></summary>

```python
def process_epoch(self, images, joints):
    self.model.train()
    T = images.shape[1] - 1
    state = None
    total_loss = 0.0
    n_windows = 0

    for start in range(0, T, self.window_step):
        end = min(start + self.window_step, T)
        yi_list, yv_list = [], []

        for t in range(start, end):
            y_img, y_jnt, _, _, state = self.model(
                images[:, t], joints[:, t], state)
            yi_list.append(y_img)
            yv_list.append(y_jnt)

        # ★ ウィンドウ間で detach
        if end != T:
            state = tuple(s.detach() for s in state)

        yi_hat = torch.stack(yi_list, dim=1)
        yv_hat = torch.stack(yv_list, dim=1)

        loss = nn.MSELoss()(yi_hat, images[:, start+1:end+1]) + \
               nn.MSELoss()(yv_hat, joints[:, start+1:end+1])
        total_loss += loss.item()
        n_windows += 1

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    return total_loss / max(n_windows, 1)
```

</details>

---
## まとめ

| 項目 | 内容 |
|:---|:---|
| **SARNN** | SpatialSoftmax で画像から注意点座標を抽出、解釈可能な予測 |
| **損失関数** | 画像損失 + 関節損失 + 注意点損失（アニーリング付き） |
| **Full BPTT** | 系列全体で逆伝播、短い系列向き |
| **Truncated BPTT** | ウィンドウ分割 + state detach、長い系列向き |
| **可視化** | 入力画像、予測画像、注意点オーバーレイ、関節角度プロット |

次回: 第9回 HSARNN → 階層型SARNNによるマルチモーダル学習